# M12 lab: model a city's energy system

*Energy Systems, Step by Step* · Modeler tier · lab for lesson M12 · about 60 minutes, plus 120 minutes of practice at the end.

You will rebuild the lesson's engine in Python and solve it with the open-source HiGHS solver through `scipy.optimize.linprog`:

1. Prairie City: three districts, eight representative days, one substation each.
2. The hub as a linear program: balances, the substation's ceiling, rooftop solar, charging, batteries, the heat network.
3. Run 2025, then electrify heat and cars, and read the substations' duals.
4. Add flexibility and watch the upgrades shrink.
5. Let the LP choose the building shares and sizes under a CO₂ cap, and read the cap's dual.
6. The screening household layer, after the LP.
7. Your own plan (the Modeler version of capstone C3).

**How to run it.** Put this notebook, `prairie-city.json` and `e3-city-hub.json` (download them from the lesson page) in one folder. In Google Colab: *File → Upload notebook*, then upload the two data files with the folder icon on the left, then *Runtime → Run all*. Locally: Python 3.9 or later with `numpy` and `scipy` 1.9 or later. Everything runs in under a minute.

**The data.** Prairie City is fictional; its numbers are illustrative. It is anchored to Kansas's 2025 prices (EIA), Wichita's weather normals (NOAA), SPP's 2025 hourly shapes (EIA-930) and eGRID's 2023 SPP North CO₂ rate (EPA). Heat pumps and rooftop solar cost what EIA says (2022 dollars, converted), batteries what NREL says (2024 dollars, converted); retrofits, the heat network, its store and substation upgrades are set for teaching. All money in 2025 dollars.

**What the asserts check.** The busiest hours, upgrades, energy bought, CO₂, yearly cost, the duals and the LP's chosen shares, against the lesson's reference numbers (`e3-city-hub.json`, `precomputed.m12.ref`, computed by the page's engine).

In [ ]:
import json
import math
from pathlib import Path

import numpy as np
import scipy
from scipy.optimize import linprog
from scipy.sparse import coo_matrix

major, minor = (int(x) for x in scipy.__version__.split(".")[:2])
assert (major, minor) >= (1, 9), "this lab needs SciPy 1.9 or later (pip install -U scipy)"
for f in ("prairie-city.json", "e3-city-hub.json"):
    assert Path(f).exists(), f + " not found: put it next to this notebook (in Colab, upload it with the folder icon)."
CITY = json.load(open("prairie-city.json", encoding="utf-8"))
CFG = json.load(open("e3-city-hub.json", encoding="utf-8"))
REF = CFG["precomputed"]["m12"]["ref"]
print(CITY["about"]["line"])
print("NumPy", np.__version__, "· SciPy", scipy.__version__)

## 1. Prairie City, prepared

- **Days.** Eight representative days of 2025; `weight` is how many days each stands for (they add to 365). The cold design day counts 3.
- **Heat.** `Q[k][h]` is the heat a district needs (MW of heat): degree-hours below 18.3 °C times its heat-loss rate.
- **COP.** The heat pump's heat per unit of power: an illustrative curve, 144 ÷ (45 − T), through DOE's cold-climate challenge target of 2.4 at −15 °C (specification, 24 September 2021); below −23 °C resistance backup is assumed (COP 1). Not a device's test data.
- **Limits.** Each substation's limit is the district's 2025 busiest hour × 1.15, rounded up to the next 10 MW (a teaching simplification).
- **Weather.** Synthetic: NOAA's 1991–2020 normals for Wichita on SPP's 2025 days, with a cold design day and a heat wave set for teaching.
- **The full year.** The lesson page's full-year check runs the plans again on all 8,760 hours of 2025, by the same rule (`CITY["fullYear"]`: its constants and sums; the hours themselves are not in the file). This notebook stays on the eight days.
- **Prices.** Each district buys at its 2025 blend of Kansas's residential and commercial electricity prices; gas and propane at 2025 prices.
- **CO₂.** Electricity at eGRID's 2023 SPP North rate plus 5% distribution losses; fuels at EIA's coefficients.

In [ ]:
u, pr, em = CITY["units"], CITY["prices"], CITY["emissions"]
MM = u["mwhPerMMBtu"]                                     # MWh in one MMBtu
GAL_PER_MWH = u["btuPerKWh"] * 1000 / u["btuPerGal"]["propane"]
DAYS = CITY["days"]; K = len(DAYS); W = [d["weight"] for d in DAYS]
assert sum(W) == 365

def cop_at(T, c=CITY["cop"]):
    return 1.0 if T < c["cutoutC"] else c["anchorCOP"] * (c["supplyC"] - c["anchorC"]) / (c["supplyC"] - T)

COP = [[cop_at(T) for T in d["temp"]] for d in DAYS]
SHAPE = [d["demandShape"] for d in DAYS]
PRICE = dict(res=pr["electricityCentsPerKWh"]["residential"] * 10, com=pr["electricityCentsPerKWh"]["commercial"] * 10,
             ind=pr["electricityCentsPerKWh"]["industrial"] * 10, gasRes=pr["gasPerMMBtu"]["residential"] / MM,
             gasCom=pr["gasPerMMBtu"]["commercial"] / MM, propMWh=pr["propanePerGal"] * GAL_PER_MWH, gasoline=pr["gasolinePerGal"])
EF = dict(grid=em["gridLbPerMWh"]["sppNorth"] / 2204.62262, loss=CITY["supply"]["gridLossShare"], gas=em["kgCO2"]["gasPerMMBtu"] / 1000 / MM,
          propGal=em["kgCO2"]["propanePerGal"] / 1000, gasoline=em["kgCO2"]["gasolinePerGal"] / 1000)
HOURS = sum(w * 24 for w in W)
ROOF_K = CITY["supply"]["rooftopCF"] / (sum(W[k] * sum(DAYS[k]["solar"]) for k in range(K)) / HOURS)
SUN = [[v * ROOF_K for v in d["solar"]] for d in DAYS]          # rooftop output per MW of panels
TR = CITY["transport"]
EV_DAY = TR["milesPerVehicle"] / 365 * TR["evKWhPerMile"] / 1000  # MWh per EV per day
GAL_PER_CAR = TR["milesPerVehicle"] / TR["mpg"]
ETA = CITY["homes"]["furnaceEfficiency"]
SYS = ["fuel", "res", "hp", "dh"]

DIST = []
for x in CITY["districts"]:
    Q = [d["districts"][x["key"]]["heat"] for d in DAYS]; B = [d["districts"][x["key"]]["base"] for d in DAYS]
    Qann = sum(W[k] * Q[k][h] for k in range(K) for h in range(24))
    Ehp = sum(W[k] * Q[k][h] / COP[k][h] for k in range(K) for h in range(24))
    sh = x["heat"]["share"]; fuel0 = sh["gas"] + sh["propane"]; Qd = x["heat"]["residentialMWh"] + x["heat"]["commercialMWh"]
    am = x["annualMWh"]; resMWh = am["residentialBase"] + am["residentialHeat"] + am["ev"]; comMWh = am["commercialBase"] + am["commercialHeat"] + am["farm"]
    g = x["gasMMBtu"]
    DIST.append(dict(key=x["key"], label=x["label"], Q=Q, B=B, Qann=Qann, eps=Ehp / Qann,
                     s0=dict(fuel=fuel0, res=sh["resistance"], hp=sh["heatPump"], dh=0.0), pf=sh["propane"] / fuel0 if fuel0 else 0.0,
                     H=x["households"], homeEq=x["households"] * Qd / x["heat"]["residentialMWh"], rho=x["heat"]["residentialMWh"] / Qd,
                     L=x["substationMW"], pv0=x["rooftopSolarMW"], evs0=x["evs"], ldv=x["lightDutyVehicles"],
                     pEl=(resMWh * PRICE["res"] + comMWh * PRICE["com"]) / (resMWh + comMWh),
                     pGas=(g["residential"] * PRICE["gasRes"] + g["commercial"] * PRICE["gasCom"]) / (g["residential"] + g["commercial"]),
                     otherGas=(g["residential"] + g["commercial"]) * MM - Qd * sh["gas"] / ETA,
                     otherProp=x["propaneGal"] / GAL_PER_MWH - Qd * sh["propane"] / ETA))
PV_SHARE = [D["pv0"] / sum(E["pv0"] for E in DIST) for D in DIST]
BAT_SHARE = [D["L"] / sum(E["L"] for E in DIST) for D in DIST]
for D in DIST:
    print(f'{D["label"]:13} limit {D["L"]:4} MW · heat {D["Qann"]:9,.0f} MWh a year · seasonal heat-pump power per heat {D["eps"]:.3f} · price ${D["pEl"]:.2f}/MWh')

## 2. A plan, and what each item costs a year

A plan is the lesson's sliders: the share of furnaces and resistance heaters switched to heat pumps (`hp`), buildings retrofitted (`ret`), the core's heat on a network (`dh`) with its plant heat pump (`dhp`, MW of heat) and store (`sto`, MWh), electric cars (`ev`, % of cars) and how they charge (`evm`), rooftop solar added (`pv`, MW), batteries (`bat`, MW), whether upgrades are allowed (`up`) and at what cost (`upc`, $/kW), the heat a retrofit saves (`save`, %), its cost (`rc`, $ a home) and the network's (`dhc`, $ a home).

Every up-front cost becomes a yearly amount: cost × the capital recovery factor at 4% real over the item's life, plus upkeep.

**Building shares.** Each district's heat is split into shares `y[e][s]`: envelope `e` (0 as built, 1 retrofitted) times system `s`. Heating power is then a sum of share × heat ÷ COP: linear, even when the LP chooses the shares.

In [ ]:
def crf(r, n): return r / (1 - (1 + r) ** -n)
def yearly(capex, life, rate=CFG["finance"]["rate"], fom_share=0.0, fom=0.0): return capex * crf(rate, life) + fom_share * capex + fom

DEFAULT = dict(hp=0, ret=0, dh=0, dhp=0, sto=0, ev=1, evm="home", pv=0, bat=0, up="yes", upc=CFG["upgrade"]["capex"]["def"],
               save=CFG["retrofit"]["save"]["def"], rc=CFG["retrofit"]["cost"]["def"], dhc=CFG["dh"]["network"]["cost"]["def"], rule="even")

def sys_of(s):
    c, dh = CFG, CFG["dh"]
    S = dict(s=s, hp=s["hp"] / 100, ret=s["ret"] / 100, save=s["save"] / 100, dh=s["dh"] / 100, dhp=s["dhp"] if s["dh"] > 0 else 0,
             sto=s["sto"] if s["dh"] > 0 else 0, evm=s["evm"], up=s["up"] == "yes", pvAdd=s["pv"], batAdd=s["bat"], rule=s["rule"])
    S["a"] = dict(hp=yearly(c["heatPump"]["capex"], c["heatPump"]["life"]), ret=yearly(s["rc"], c["retrofit"]["life"]),
                  net=yearly(s["dhc"], dh["network"]["life"]), plant=yearly(dh["plant"]["capex"]["def"] * 1000, dh["plant"]["life"]),
                  store=yearly(dh["store"]["capex"]["def"], dh["store"]["life"]),
                  pv=yearly(c["rooftop"]["capex"] * 1000, c["rooftop"]["life"], fom=c["rooftop"]["fom"] * 1000),
                  bat=yearly(c["battery"]["capex"] * 1000, c["battery"]["life"], fom_share=c["battery"]["fomShare"]),
                  up=yearly(s["upc"] * 1000, c["upgrade"]["life"]))
    S["d"] = []
    for i, D in enumerate(DIST):
        s0, x, cc, r = D["s0"], (S["dh"] if i == 0 else 0.0), S["hp"], S["ret"]
        after = dict(fuel=(1 - x) * s0["fuel"] * (1 - cc), res=(1 - x) * s0["res"] * (1 - cc), hp=(1 - x) * (s0["hp"] + cc * (s0["fuel"] + s0["res"])), dh=x)
        y = [{k: (1 - r) * after[k] for k in SYS}, {k: r * after[k] for k in SYS}]
        evs = max(D["evs0"], int(math.floor(s["ev"] / 100 * D["ldv"] + 0.5)))   # JavaScript's Math.round
        S["d"].append(dict(after=after, y=y, evs=evs, E=evs * EV_DAY, pv=D["pv0"] + S["pvAdd"] * PV_SHARE[i], pvAdd=S["pvAdd"] * PV_SHARE[i], bat=S["batAdd"] * BAT_SHARE[i]))
    return S

print("yearly cost of 1 MW of upgrade: $%s; of one heat pump: $%s" % (format(round(sys_of(DEFAULT)["a"]["up"]), ","), format(round(sys_of(DEFAULT)["a"]["hp"]), ",")))

## 3. The linear program

The model builder below writes one variable per decision and one row per rule, then hands them to `linprog`:

- **balance** (row `b`), each district-hour: bought + rooftop used + battery out − battery in − charging − the plant's power − heating power = base load. Its dual is the marginal cost of one more MWh there.
- **substation** (row `l`): bought − upgrade ≤ limit. Its dual is what one more MW of room is worth in that hour.
- **tie rule**: small stated costs on each district's busiest hour, the three districts' together, battery and store output and the timing of managed charging; they pick one schedule among equal-cost ones and are left out of the reported costs.
- **choose mode** (`size=True`): the building shares and the sizes become decisions with their yearly costs, and an optional CO₂ cap.

In [ ]:
class Model:
    def __init__(self):
        self.names, self.lb, self.ub, self.cost, self.idx, self.rows = [], [], [], [], {}, []
    def var(self, name, lb=0.0, ub=math.inf, cost=0.0):
        self.idx[name] = len(self.names); self.names.append(name); self.lb.append(lb); self.ub.append(ub); self.cost.append(cost); return name
    def row(self, name, terms, lo=None, hi=None):
        self.rows.append((name, terms, lo, hi))
    def solve(self):
        n = len(self.names); eq, ub, beq, bub, neq, nub = [], [], [], [], [], []
        for name, terms, lo, hi in self.rows:
            if lo is not None and hi is not None and lo == hi:
                eq.append(terms); beq.append(lo); neq.append(name)
            else:
                if hi is not None: ub.append(terms); bub.append(hi); nub.append((name, 1))
                if lo is not None: ub.append([(v, -c) for v, c in terms]); bub.append(-lo); nub.append((name, -1))
        def mat(rs):
            r, c, v = [], [], []
            for i, terms in enumerate(rs):
                for name, coef in terms: r.append(i); c.append(self.idx[name]); v.append(coef)
            return coo_matrix((v, (r, c)), shape=(len(rs), n)).tocsr()
        res = linprog(self.cost, A_ub=mat(ub) if ub else None, b_ub=bub or None, A_eq=mat(eq) if eq else None, b_eq=beq or None,
                      bounds=list(zip(self.lb, [None if u == math.inf else u for u in self.ub])), method="highs")
        out = dict(status=res.status, message=res.message, objective=res.fun)
        if res.status == 0:
            out["x"] = {nm: res.x[i] for i, nm in enumerate(self.names)}
            dual = {}
            for (nm, sgn), m in zip(nub, res.ineqlin.marginals): dual[nm] = dual.get(nm, 0.0) + sgn * m
            for nm, m in zip(neq, res.eqlin.marginals): dual[nm] = m
            out["dual"] = dual        # d objective / d right-hand side, as the browser's LX.solver reports it
        return out

def build(S, size=False, cap=None, ret_max=1.0, dh_max=0.5, pv_max=400, bat_max=200):
    T, dh, M = CFG["tie"], CFG["dh"], Model()
    eff, keep, net, cop_p = CFG["battery"]["rte"], 1 - dh["store"]["lossPerHour"], 1 - dh["loss"], dh["plant"]["cop"]
    co2, ef_el = [], EF["grid"] * (1 + EF["loss"])
    Y = []
    for i, D in enumerate(DIST):
        out = [{}, {}]
        for e in (0, 1):
            for k in SYS:
                if k == "dh" and i != 0: continue
                fix, f = S["d"][i]["y"][e][k] * 100, (1 - S["save"] * e) * D["Qann"] / 100     # shares in percent
                cost = f / ETA * ((1 - D["pf"]) * D["pGas"] + D["pf"] * PRICE["propMWh"]) if k == "fuel" else 0.0
                if size:
                    if e == 1: cost += S["a"]["ret"] * D["homeEq"] / 100
                    if k == "dh": cost += S["a"]["net"] * D["homeEq"] / 100
                    out[e][k] = M.var(f"y{i}{e}{k}", 0, math.inf, cost)
                else:
                    out[e][k] = M.var(f"y{i}{e}{k}", fix, fix, cost)
                if k == "fuel": co2.append((out[e][k], f / ETA * ((1 - D["pf"]) * EF["gas"] + D["pf"] * GAL_PER_MWH * EF["propGal"])))
        if size:
            M.row(f"yall{i}", [(v, 1) for e in (0, 1) for v in out[e].values()], 100, 100)
            M.row(f"yfuel{i}", [(out[0]["fuel"], 1), (out[1]["fuel"], 1)], None, D["s0"]["fuel"] * 100)
            M.row(f"yres{i}", [(out[0]["res"], 1), (out[1]["res"], 1)], None, D["s0"]["res"] * 100)
            M.row(f"yret{i}", [(v, 1) for v in out[1].values()], None, ret_max * 100)
            if i == 0: M.row("ydh", [(out[0]["dh"], 1), (out[1]["dh"], 1)], None, dh_max * 100)
            zh = M.var(f"zhp{i}", 0, math.inf, S["a"]["hp"] * D["homeEq"] / 100)
            M.row(f"zhp{i}", [(zh, 1), (out[0]["hp"], -1), (out[1]["hp"], -1)], -D["s0"]["hp"] * 100, None)
        Y.append(out)
    pvA = M.var("pvAdd", 0, pv_max, S["a"]["pv"]) if size else None
    batA = M.var("batAdd", 0, bat_max, S["a"]["bat"]) if size else None
    dh_on = (dh_max > 0) if size else S["dh"] > 0
    dhpV = M.var("dhp", 0, 250, S["a"]["plant"]) if (dh_on and size) else None
    stoV = M.var("sto", 0, 2000, S["a"]["store"]) if (dh_on and size) else None
    upg = [M.var(f"upg{i}", 0, math.inf if S["up"] else 0, S["a"]["up"]) for i in range(len(DIST))]
    pk = [M.var(f"pk{i}", 0, math.inf, T["peak"]) for i in range(len(DIST))]
    pkC = M.var("pkC", 0, math.inf, T["cityPeak"])
    for k in range(K):
        w = W[k]
        for h in range(24):
            tag, city = f"{k}_{h}", [(pkC, -1)]
            for i, D in enumerate(DIST):
                sd, t = S["d"][i], f"{i}_{k}_{h}"
                imp = M.var("i" + t, 0, math.inf, w * D["pEl"]); bal = [(imp, 1)]; rhs = D["B"][k][h]
                co2.append((imp, w * ef_el))
                if size or sd["pv"] > 0:
                    pvu = M.var("s" + t, 0, math.inf if size else sd["pv"] * SUN[k][h]); bal.append((pvu, 1))
                    if size: M.row("sm" + t, [(pvu, 1), (pvA, -SUN[k][h] * PV_SHARE[i])], None, D["pv0"] * SUN[k][h])
                for e in (0, 1):
                    q = (1 - S["save"] * e) * D["Q"][k][h] / 100
                    if q > 0: bal += [(Y[i][e]["res"], -q), (Y[i][e]["hp"], -q / COP[k][h])]
                if S["evm"] == "managed":
                    ev = M.var("v" + t, 0, sd["E"] / CFG["ev"]["minHours"], T["ev"] * SHAPE[k][h] / 1000 * w); bal.append((ev, -1))
                    if h == 23: M.row(f"ve{i}_{k}", [(f"v{i}_{k}_{hv}", 1) for hv in range(24)], sd["E"], sd["E"])
                else:
                    rhs += sd["E"] * TR["evProfile"][h]
                if size or sd["bat"] > 0:
                    bm = math.inf if size else sd["bat"]
                    bi, bo = M.var("bc" + t, 0, bm), M.var("bd" + t, 0, bm, T["wear"] * w)
                    bq = M.var("bq" + t, 0, math.inf if size else sd["bat"] * CFG["battery"]["hours"])
                    bal += [(bo, 1), (bi, -1)]
                    M.row("bs" + t, [(bq, 1), (f"bq{i}_{k}_{(h + 23) % 24}", -1), (bi, -eff), (bo, 1)], 0, 0)
                    if size:
                        sb = BAT_SHARE[i]
                        M.row("bci" + t, [(bi, 1), (batA, -sb)], None, 0); M.row("bco" + t, [(bo, 1), (batA, -sb)], None, 0)
                        M.row("bcq" + t, [(bq, 1), (batA, -sb * CFG["battery"]["hours"])], None, 0)
                if i == 0 and dh_on:
                    hp = M.var("dp" + tag, 0, math.inf if size else S["dhp"], w * (PRICE["ind"] - D["pEl"]) / cop_p)
                    boil = M.var("db" + tag, 0, math.inf, w * PRICE["gasCom"] / dh["plant"]["boilerEff"])
                    co2.append((boil, w * EF["gas"] / dh["plant"]["boilerEff"]))
                    bal.append((hp, -1 / cop_p))
                    if size: M.row("dpc" + tag, [(hp, 1), (dhpV, -1)], None, 0)
                    heat = [(hp, 1), (boil, 1)]
                    if size or S["sto"] > 0:
                        mh = dh["store"]["minHours"]
                        si = M.var("ti" + tag, 0, math.inf if size else S["sto"] / mh)
                        so = M.var("to" + tag, 0, math.inf if size else S["sto"] / mh, T["storeWear"] * w)
                        sq = M.var("tq" + tag, 0, math.inf if size else S["sto"])
                        heat += [(so, 1), (si, -1)]
                        M.row("ts" + tag, [(sq, 1), (f"tq{k}_{(h + 23) % 24}", -keep), (si, -1), (so, 1)], 0, 0)
                        if size:
                            M.row("tci" + tag, [(si, 1), (stoV, -1 / mh)], None, 0); M.row("tco" + tag, [(so, 1), (stoV, -1 / mh)], None, 0)
                            M.row("tcq" + tag, [(sq, 1), (stoV, -1)], None, 0)
                    for e in (0, 1):
                        q = (1 - S["save"] * e) * D["Q"][k][h] / net / 100
                        if q > 0: heat.append((Y[0][e]["dh"], -q))
                    M.row("dh" + tag, heat, 0, 0)
                M.row("b" + t, bal, rhs, rhs)
                M.row("l" + t, [(imp, 1), (upg[i], -1)], None, D["L"])
                M.row("p" + t, [(imp, 1), (pk[i], -1)], None, 0)
                city.append((imp, 1))
            M.row("c" + tag, city, None, 0)
    if cap is not None:
        fixed = sum(D["otherGas"] * EF["gas"] + D["otherProp"] * GAL_PER_MWH * EF["propGal"] + (D["ldv"] - S["d"][i]["evs"]) * GAL_PER_CAR * EF["gasoline"] for i, D in enumerate(DIST))
        M.row("cap", co2, None, cap - fixed)
    M.dh_on, M.size = dh_on, size
    return M

M = build(sys_of(DEFAULT))
print(f"2025 as it is: {len(M.names):,} decisions, {len(M.rows):,} rules")

## 4. Reading a solved plan, and the year's accounts

`run(plan)` solves and returns the busiest hours, upgrades, energy bought, the year's cost and CO₂ (the same accounting as the browser: energy at 2025 prices, equipment and upgrades as yearly payments, the tie rule's costs left out), and the summed substation duals.

In [ ]:
def run(plan, size=False, cap=None, **kw):
    S = sys_of(plan); M = build(S, size=size, cap=cap, **kw); r = M.solve()
    assert r["status"] == 0, r["message"]
    x, du = r["x"], r["dual"]; val = lambda n: x.get(n, 0.0)
    out = dict(S=S, r=r, x=x, peaks=[val(f"pk{i}") for i in range(3)], upg=[val(f"upg{i}") for i in range(3)])
    y = [[{k: (val(f"y{i}{e}{k}") / 100 if not (k == "dh" and i != 0) else 0.0) for k in SYS} for e in (0, 1)] for i in range(3)]
    sizes = dict(pvAdd=val("pvAdd"), batAdd=val("batAdd"), dhp=val("dhp"), sto=val("sto")) if size else dict(pvAdd=S["pvAdd"], batAdd=S["batAdd"], dhp=S["dhp"], sto=S["sto"])
    wsum = lambda f: sum(W[k] * f(k, h) for k in range(K) for h in range(24))
    money = dict(elec=0.0, gas=0.0, prop=0.0, gasoline=0.0, cap=0.0); co2 = 0.0; ef_el = EF["grid"] * (1 + EF["loss"]); imp_all = 0.0; d_out = []
    for i, D in enumerate(DIST):
        imp = wsum(lambda k, h: val(f"i{i}_{k}_{h}")); imp_all += imp
        plant = wsum(lambda k, h: val(f"dp{k}_{h}") / CFG["dh"]["plant"]["cop"]) if (i == 0 and M.dh_on) else 0.0
        fuel = sum((1 - S["save"] * e) * D["Qann"] * y[i][e]["fuel"] for e in (0, 1))
        gas, prop = fuel * (1 - D["pf"]) / ETA + D["otherGas"], fuel * D["pf"] / ETA + D["otherProp"]
        gasoline = (D["ldv"] - S["d"][i]["evs"]) * GAL_PER_CAR
        share = {k: y[i][0][k] + y[i][1][k] for k in SYS}; ret = sum(y[i][1].values())
        money["elec"] += imp * D["pEl"] - plant * (D["pEl"] - PRICE["ind"]); money["gas"] += gas * D["pGas"]
        money["prop"] += prop * PRICE["propMWh"]; money["gasoline"] += gasoline * PRICE["gasoline"]
        money["cap"] += max(0.0, share["hp"] - D["s0"]["hp"]) * D["homeEq"] * S["a"]["hp"] + ret * D["homeEq"] * S["a"]["ret"] + share["dh"] * D["homeEq"] * S["a"]["net"] + out["upg"][i] * S["a"]["up"]
        co2 += imp * ef_el + gas * EF["gas"] + prop * GAL_PER_MWH * EF["propGal"] + gasoline * EF["gasoline"]
        d_out.append(dict(imp=imp, share=share, ret=ret, value=-sum(du[f"l{i}_{k}_{h}"] for k in range(K) for h in range(24))))
    boil = 0.0
    if M.dh_on:
        boil = wsum(lambda k, h: val(f"db{k}_{h}")) / CFG["dh"]["plant"]["boilerEff"]
        money["gas"] += boil * PRICE["gasCom"]; co2 += boil * EF["gas"]
    money["cap"] += sizes["dhp"] * S["a"]["plant"] + sizes["sto"] * S["a"]["store"] + sizes["pvAdd"] * S["a"]["pv"] + sizes["batAdd"] * S["a"]["bat"]
    # the heat network's average cost per MWh of heat (the household layer bills district heat at it): network, plant,
    # store, the boiler's gas and the plant's power at the industrial price
    D0 = DIST[0]; dh_heat = sum((1 - S["save"] * e) * D0["Qann"] * y[0][e]["dh"] for e in (0, 1))
    plant_el = wsum(lambda k, h: val(f"dp{k}_{h}") / CFG["dh"]["plant"]["cop"]) if M.dh_on else 0.0
    dh_cost = d_out[0]["share"]["dh"] * D0["homeEq"] * S["a"]["net"] + sizes["dhp"] * S["a"]["plant"] + sizes["sto"] * S["a"]["store"] + boil * PRICE["gasCom"] + plant_el * PRICE["ind"]
    out.update(y=y, sizes=sizes, d=d_out, imp=imp_all, co2=co2, cost=sum(money.values()), money=money, capDual=-du["cap"] if cap is not None else None,
               dh_price=dh_cost / dh_heat if dh_heat > 0 else 0.0)
    return out

def show(name, o):
    print(f"{name}: busiest hours " + ", ".join(f'{D["key"]} {o["peaks"][i]:.1f}/{D["L"]} MW' for i, D in enumerate(DIST)) +
          f' · upgrades {sum(o["upg"]):.1f} MW · CO2 {o["co2"]:,.0f} t · cost ${o["cost"] / 1e6:,.1f}M a year')

def check(o, ref):
    assert all(abs(a - b) < 0.05 for a, b in zip(o["peaks"], ref["peaks"])), (o["peaks"], ref["peaks"])
    assert all(abs(a - b) < 0.05 for a, b in zip(o["upg"], ref["upg"])), (o["upg"], ref["upg"])
    assert abs(o["co2"] - ref["co2"]) < 2 and abs(o["imp"] - ref["imp"]) < 2, (o["co2"], ref["co2"], o["imp"], ref["imp"])
    assert abs(o["cost"] - ref["cost"]) < 10, (o["cost"], ref["cost"])

PLANS = dict(today=dict(DEFAULT), electrify=dict(DEFAULT, hp=80, ev=50), flex=dict(DEFAULT, hp=80, ev=50, evm="managed", bat=40))
RUNS = {}
for key, plan in PLANS.items():
    RUNS[key] = run(plan); show(key, RUNS[key]); check(RUNS[key], REF["runs"][key])
print("ok: the three runs match the browser's")

## 5. The substations' duals

Where the LP buys an upgrade, one more MW of limit saves one MW of upgrade, so the substation row's duals, summed over the hours they bind, equal the upgrade's yearly cost per MW. Check it, then check the value as a cost slope: move the suburbs' limit 0.1 MW up and down.

In [ ]:
E = RUNS["electrify"]
for i, D in enumerate(DIST):
    print(f'{D["label"]:13} upgrade {E["upg"][i]:5.1f} MW · summed duals ${E["d"][i]["value"]:,.0f} per MW a year (browser ${REF["runs"]["electrify"]["value"][i]:,.0f})')
    assert abs(E["d"][i]["value"] - REF["runs"]["electrify"]["value"][i]) < 1
    if E["upg"][i] > 0.05: assert abs(E["d"][i]["value"] - E["S"]["a"]["up"]) < 1

def objective_with_limit(plan, i, dL):
    old = DIST[i]["L"]; DIST[i]["L"] = old + dL
    try: return build(sys_of(plan)).solve()["objective"]
    finally: DIST[i]["L"] = old

z0 = E["r"]["objective"]; up_ = (z0 - objective_with_limit(PLANS["electrify"], 1, 0.1)) / 0.1; dn_ = (objective_with_limit(PLANS["electrify"], 1, -0.1) - z0) / 0.1
print(f"suburbs: cost slope up ${up_:,.0f}, down ${dn_:,.0f} per MW a year")
assert abs(up_ - E["d"][1]["value"]) < 50 and abs(dn_ - E["d"][1]["value"]) < 50

## 6. Let the LP choose under a CO₂ cap

Now the building shares, rooftop solar, batteries and the heat network's plant and store are decisions, at their yearly costs; half the cars are electric with managed charging. Cap the districts' CO₂ at 80% of 2025 and read the cap's dual: the cost of cutting the last tonne, under this model's assumptions.

In [ ]:
base_co2 = RUNS["today"]["co2"]; assert abs(base_co2 - REF["base"]) < 2
CH = run(dict(DEFAULT, ev=50, evm="managed"), size=True, cap=base_co2 * REF["cap"] / 100, ret_max=0.6, dh_max=0.4)
print(f'cap {REF["cap"]}% of 2025: CO2 {CH["co2"]:,.0f} t · the cap\'s dual ${CH["capDual"]:,.0f} per t (browser ${REF["runs"]["choose"]["capDual"]:,.0f})')
for i, D in enumerate(DIST):
    s = CH["d"][i]["share"]; print(f'  {D["label"]:13} heat pumps {s["hp"]:.0%} · retrofitted {CH["d"][i]["ret"]:.0%} · district heat {s["dh"]:.0%} · furnaces {s["fuel"]:.0%}')
print(f'  rooftop solar added {max(0.0, CH["sizes"]["pvAdd"]):.1f} MW · batteries {max(0.0, CH["sizes"]["batAdd"]):.1f} MW')
check(CH, REF["runs"]["choose"])
assert abs(CH["capDual"] - REF["runs"]["choose"]["capDual"]) < 0.5
assert all(abs(CH["d"][i]["share"]["hp"] - REF["runs"]["choose"]["hp"][i]) < 1e-3 for i in range(3))
assert abs(CH["sizes"]["pvAdd"] - REF["runs"]["choose"]["pvAdd"]) < 0.05
print("ok: the LP's choice matches the browser's")

## 7. The screening household layer, after the LP

Nothing below changes the LP's plan. For each measure (heating switched, retrofits, rooftop solar) a district's change is spread over the five income groups: each group's *potential* is its eligible homes times its use index, and the rule fills it (`even`: every group at the same rate; `low` or `high`: lowest or highest income first). Cars come from a table of the 2025 non-electric cars' gasoline by district and group, fitted to both totals, so a new EV replaces only gasoline a group has. Then lesson A13's accounting layer (`allocate`, its per-kWh design) adds the households' share of the shared costs (upgrades, batteries) on every kWh; bills are at 2025 residential prices; district heat is billed at its average cost. Energy burden is home energy bills ÷ income (DOE LEAD's definition), which leaves transport out: the charging the plan adds goes with motor fuel. It is a screening estimate, not a prediction.

In [ ]:
GROUPS, HH = CITY["groups"], CITY["householdsByGroup"]["households"]

def spread(room, N, rule):
    tot = sum(room); n = [0.0] * len(room)
    if N <= 0 or tot <= 0: return n
    N = min(N, tot)
    if rule == "even": return [r / tot * N for r in room]
    order = list(range(len(room)))[::-1] if rule == "high" else list(range(len(room)))
    for j in order: a = min(room[j], N); n[j] = a; N -= a
    return n

# the 2025 cars' gasoline by district (rows) and group (columns), fitted so rows add to each district's non-electric
# cars and columns to each group's gasoline (iterative proportional fitting), as the browser's fuelMatrix
def fuel_matrix():
    rows = [(D["ldv"] - D["evs0"]) * GAL_PER_CAR for D in DIST]
    cols = [g["households"] * g["use"]["gasolineGal"] for g in GROUPS]; kc = sum(rows) / sum(cols); cols = [c * kc for c in cols]
    M = [[HH[i][j] * GROUPS[j]["index"]["gasoline"] for j in range(len(GROUPS))] for i in range(3)]
    for _ in range(500):
        M = [[v * rows[i] / sum(r) for v in r] for i, r in enumerate(M)]
        for j in range(len(GROUPS)):
            sj = sum(M[i][j] for i in range(3))
            for i in range(3): M[i][j] *= cols[j] / sj
    return M, cols
FM, FCOLS = fuel_matrix()

def households(o, o0, rule):
    S, ng = o["S"], len(GROUPS)
    dEl, dGas, dProp, dGal, dDh, dEv = [[0.0] * ng for _ in range(6)]
    for i, D in enumerate(DIST):
        H, Hd, a, a0 = HH[i], sum(HH[i]), o["d"][i], o0["d"][i]
        Qr = D["rho"] * D["Qann"]; s0, sh, ret = D["s0"], a["share"], a["ret"]
        mix = lambda s: dict(el=Qr * (s["res"] + s["hp"] * D["eps"]), gas=Qr * s["fuel"] * (1 - D["pf"]) / ETA, prop=Qr * s["fuel"] * D["pf"] / ETA, dh=Qr * s["dh"])
        m0, m1 = mix(s0), mix(sh)
        changed = 1 - sum(min(sh[k], s0[k]) for k in SYS)
        meas = [(changed * Hd, max(changed, 1 - s0["hp"]), {c: m1[c] - m0[c] for c in m1}, "gas"),
                (ret * Hd, 1.0, {c: -S["save"] * ret * m1[c] for c in m1}, "gas")]
        pv_used = lambda oo: sum(W[k] * oo["x"].get(f"s{i}_{k}_{h}", 0.0) for k in range(K) for h in range(24))
        pv_homes = o["sizes"]["pvAdd"] * PV_SHARE[i] / 0.005           # a 5 kW system a home (set for teaching)
        meas.append((min(pv_homes, Hd), 1.0, dict(el=-(pv_used(o) - pv_used(o0))), "electricity"))
        for n, cap_, dl, ix in meas:
            if n <= 0 or cap_ <= 0: continue
            pot = [cap_ * H[g] * GROUPS[g]["index"][ix] for g in range(ng)]
            got = spread(pot, min(1.0, n / (cap_ * Hd)) * sum(pot), rule); A = sum(got)
            if A <= 0: continue
            for c, arr in (("el", dEl), ("gas", dGas), ("prop", dProp), ("dh", dDh)):
                for g in range(ng): arr[g] += dl.get(c, 0.0) * got[g] / A
        # the EVs: each replaces one of the district's 2025 non-electric cars, in the group that drives it
        evN = S["d"][i]["evs"] - D["evs0"]
        if evN > 0:
            cars = spread([gal / GAL_PER_CAR for gal in FM[i]], evN, rule)
            for g in range(ng): dGal[g] -= cars[g] * GAL_PER_CAR; dEv[g] += cars[g] * EV_DAY * 365
    # a group cannot cut more than it uses: the rest of a cut passes to the next groups in the rule's order (every group
    # alike: in proportion to their remaining use); each carrier's total is kept, as in the browser
    use0 = dict(el=[g["households"] * g["use"]["electricityKWh"] / 1000 for g in GROUPS], gas=[g["households"] * g["use"]["gasMMBtu"] * MM for g in GROUPS],
                prop=[g["households"] * g["use"]["propaneGal"] / GAL_PER_MWH for g in GROUPS])
    for c, arr in (("el", dEl), ("gas", dGas), ("prop", dProp)):
        assert settle(use0[c], arr, rule) >= 0, "the groups together cannot take this cut"
    meter = sum(dd["imp"] for dd in o["d"]) + sum(W[k] * o["x"].get(f"s{i}_{k}_{h}", 0.0) for i in range(3) for k in range(K) for h in range(24))
    adder = (sum(o["upg"]) * S["a"]["up"] + o["sizes"]["batAdd"] * S["a"]["bat"]) / meter     # $ per MWh on every MWh used
    rows = []
    for g, gr in enumerate(GROUPS):
        Hg, sp = gr["households"], gr["spending"]
        el = gr["use"]["electricityKWh"] / 1000 + dEl[g] / Hg                                 # the home's MWh (no new charging)
        home = sp["electricity"] + dEl[g] / Hg * PRICE["res"] + el * adder + sp["gas"] + dGas[g] / Hg * PRICE["gasRes"] + sp["propane"] + dProp[g] / Hg * PRICE["propMWh"] + dDh[g] / Hg * o["dh_price"]
        fuel = sp["motorFuel"] * max(0.0, 1 + dGal[g] / FCOLS[g]); charging = dEv[g] / Hg * (PRICE["res"] + adder)
        assert fuel >= 0 and el >= -1e-9, "a group's use below zero"
        rows.append(dict(label=gr["label"], households=Hg, income=gr["income"], home0=gr["homeEnergy"], home=home, transport0=sp["motorFuel"], transport=fuel + charging))
    agg = lambda sel, key: sum(r["households"] * r[key] for r in sel) / sum(r["households"] * r["income"] for r in sel)
    return rows, agg(rows[:3], "home"), agg(rows[3:], "home")

def settle(use, d, rule):
    ex = 0.0
    for g in range(len(use)):
        if use[g] + d[g] < 0: ex += -(use[g] + d[g]); d[g] = -use[g]
    if ex <= 1e-9: return 0.0
    moved, room = ex, [max(0.0, u + x) for u, x in zip(use, d)]
    if sum(room) < ex - 1e-9: return -1.0
    if rule == "even":
        tot = sum(room)
        for g in range(len(use)): d[g] -= ex * room[g] / tot
        return moved
    for g in (range(len(use))[::-1] if rule == "high" else range(len(use))):
        t = min(room[g], ex); d[g] -= t; ex -= t
    return moved

# an SB3 (City Planner) link for a plan: the page reads the same settings and shows its household table
def city_planner_link(plan):
    enum = dict(evm=["home", "managed"], up=["yes", "no"], rule=["even", "low", "high"], rec=["kwh", "fixed", "income"])
    order = [("hp", "h"), ("ret", "r"), ("dh", "d"), ("dhp", "p"), ("sto", "s"), ("ev", "e"), ("evm", "m"), ("pv", "v"), ("bat", "b"), ("up", "u"), ("upc", "c"), ("save", "a"), ("rc", "k"), ("dhc", "n"), ("rule", "o"), ("rec", "q")]
    p = dict(plan, rec=plan.get("rec", "kwh"))
    return "https://henryutd2021.github.io/learn/sb3-city-planner.html#sb3-v1-" + "-".join(sh + str(enum[k].index(p[k]) if k in enum else p[k]) for k, sh in order)

for key in ("electrify", "flex", "choose"):
    o = CH if key == "choose" else RUNS[key]
    rows, low, other = households(o, RUNS["today"], "even")
    print(f'{key:9}: burden below 80% of AMI {low:.2%} (browser {REF["runs"][key]["burdenLow"]:.2%}) · others {other:.2%}')
    assert abs(low - REF["runs"][key]["burdenLow"]) < 2e-5 and abs(other - REF["runs"][key]["burdenOther"]) < 2e-5
for rule in ("even", "low", "high"):
    rows, low, _ = households(CH, RUNS["today"], rule)
    print(f"rule {rule:5}: " + " · ".join(f'{r["label"].split(" of")[0]} ${r["home0"]:,.0f}→${r["home"]:,.0f}' for r in rows))
print("ok: the household layer matches the browser's; the rule moved bills, not the LP's plan")

## 8. Your plan (the Modeler version of capstone C3)

Set the district levers of your C3 plan in the cell below (the one that starts `MY = `) and run it. This lab models the three districts only: C3's industrial park, the data-center campus and the farms are on the capstone page. Hand in the printed lines with your plan summary.

In [ ]:
MY = dict(DEFAULT, hp=70, ret=40, dh=30, dhp=100, sto=800, ev=60, evm="managed", pv=100, bat=20)   # your levers
O = run(MY); show("your plan", O)
rows, low, other = households(O, RUNS["today"], MY["rule"])
dc, dm = O["co2"] - RUNS["today"]["co2"], O["cost"] - RUNS["today"]["cost"]
print(f'CO2 against 2025: {"+" if dc >= 0 else "−"}{abs(dc):,.0f} t a year; yearly cost against 2025: {"+" if dm >= 0 else "−"}${abs(dm) / 1e6:,.1f}M')
print(f"burden below 80% of area median income: {low:.2%} (screening; district heat billed at ${O['dh_price']:,.0f} per MWh of heat)")
if {k: MY[k] for k in REF["my"]["plan"]} == REF["my"]["plan"] and all(MY[k] == DEFAULT[k] for k in DEFAULT if k not in REF["my"]["plan"]):
    assert abs(low - REF["my"]["burdenLow"]) < 2e-5 and abs(O["dh_price"] - REF["my"]["dhPrice"]) < 0.01
    print("ok: this plan's household numbers match the browser's (change MY and they become yours)")
print("the same plan in the City Planner (its household table, with its assumptions):", city_planner_link(MY))

## Practice (about 120 minutes)

1. **The design day: weight against constraints.** (a) Give the cold design day a weight of 0 (move its 3 days to the winter day) and run *electrify* again. The upgrades stay: a weight scales a day's costs and CO₂ in the year's totals, but its hourly balances and substation rows still bind. (b) Now remove the day itself: build the model over the other seven days only (skip its index in the loops over `k`, with its 3 days on the winter day). Which upgrades disappear, and what does that say about planning on average days?
2. **A colder snap.** Lower the design day's temperatures by 5 °C (recompute `Q` with the district's `uaMWperC` and `COP`). How much more upgrade does *electrify* need?
3. **Hybrid heat pumps.** Add a system `"hyb"`: a heat pump that switches to the furnace below −10 °C. Write its heating power and gas use as shares, and let the LP choose it under the 80% cap. Does it replace upgrades?
4. **A tighter cap.** Trace the cap's dual from 100% down to 65% in steps of 5%. Where does it jump, and which decision changes at each jump?
5. **Who adopts first.** For the LP's choice at 80%, compare the three rules' bills for the lowest group. Write two sentences on what the rule assumes and why it is not a prediction.
6. **The other cold days.** `CITY["fullYear"]["designDays"]` lists the 3 days the design day stands for in the full year. Compare each district's `peak2025MW` and `peakAt` with `peak2025DaysMW` in `CITY["fullYear"]["districts"]`. Which day sets the year's busiest hour, and what differs from the design day there (temperature, heat or base load)? Then say in two sentences why the page's full-year check finds the eight days' upgrades a little short.